# OdomAI — Used Car Price Model Training

End-to-end pipeline that turns the raw [Craigslist Cars & Trucks dataset](https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data) into the artifacts served by the Flask backend in `backend/app.py`:

1. Load ~1.4M raw listings and keep the columns the model needs
2. Clean outliers, normalize text, and fix cross-brand model/manufacturer leaks (e.g. a Jeep Wrangler listed with manufacturer "wrangler")
3. Engineer features (`age`, `miles_per_year`) and log-transform the price target
4. Target-encode categorical columns (manufacturer/model/fuel/transmission) with `category_encoders`
5. Train an `XGBRegressor`, evaluate it, and inspect feature importances
6. Save the trained model, encoder, and column order to `models/` for the Flask API to load

**To re-run this notebook locally:** download `vehicles.csv` from the Kaggle dataset above and place it at `data/vehicles.csv` (relative to the repo root). Artifacts are written to `models/`, matching what `backend/app.py` loads at startup.

## 1. Load & clean the raw listings

Drop incomplete rows, filter unrealistic prices/mileage/years, and normalize manufacturer/model text (including fixing cars whose model name leaked into the manufacturer field).

## 2. Feature engineering, target encoding & training

Split before encoding to avoid target leakage, target-encode the categorical columns, train an XGBoost regressor on log-price, then evaluate with MAE / RMSE / R² and inspect which features drive the prediction.

In [ ]:
# ============================================
# STEP 1 — Load the data
# ============================================
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/vehicles.csv')
print("Initial shape:", df.shape)


# ============================================
# STEP 2 — Keep only core columns
# ============================================
keep_cols = ['price', 'year', 'manufacturer', 'model', 'fuel', 'odometer', 'transmission']
df = df[keep_cols]


# ============================================
# STEP 3 — Drop rows missing core fields
# ============================================
df = df.dropna(subset=keep_cols)


# ============================================
# STEP 4 — Filter outliers & junk listings
# ============================================
df = df[(df['price'] >= 1000) & (df['price'] <= 120000)]
df = df[(df['odometer'] >= 1000) & (df['odometer'] <= 300000)]
df = df[(df['year'] >= 1995) & (df['year'] <= 2024)]


import pandas as pd
import re

# ============================================
# STEP 5 — Normalize, Clean Models/Trims, and Filter Leaks
# ============================================

# 1. Lowercase and strip basic strings
for col in ['manufacturer', 'model', 'fuel', 'transmission']:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Clean manufacturer aliases
df = df[~df['manufacturer'].isin(['harley-davidson'])]
df['manufacturer'] = df['manufacturer'].replace({'land rover': 'land rover', 'rover': 'land rover'})

# 2. Re-assign manufacturer for iconic cross-brand leaked models
ICONIC_MODEL_TO_MAKE = {
    r'\bprius\b': 'toyota',
    r'\baltima\b': 'nissan',
    r'\bcivic\b': 'honda',
    r'\bcorolla\b': 'toyota',
    r'\bcamry\b': 'toyota',
    r'\baccord\b': 'honda',
    r'\bmustang\b': 'ford',
    r'\bwrangler\b': 'jeep',
    r'\bsanta fe\b': 'hyundai',
    r'\bc-?class\b|\bc300\b|\bc250\b': 'mercedes-benz',
    r'\bsorento\b': 'kia',
    r'\bgrand caravan\b|\bcaravan\b': 'dodge',
    r'\bsilverado\b': 'chevrolet',
    r'\bf-?150\b|\bf150\b': 'ford',
}

def fix_cross_brand_leaks(row):
    make, model = row['manufacturer'], row['model']
    for pattern, correct_make in ICONIC_MODEL_TO_MAKE.items():
        if re.search(pattern, model):
            return correct_make
    return make

df['manufacturer'] = df.apply(fix_cross_brand_leaks, axis=1)

# 3. Standardize Model & Trim Strings
def clean_model_name(row):
    make = row['manufacturer']
    model = row['model']
    
    if not isinstance(model, str) or not model or model == 'nan':
        return 'other'
        
    model = re.sub(r'^(benz|romeo|scion|subaru|mazda|ford|chevy|chevrolet)\s+', '', model)
    model = re.sub(r'[\&\+]', ' and ', model)
    
    # --- Standardize Messy Trim Spellings ---
    model = re.sub(r'\bex l\b|\bexl\b', 'ex-l', model)
    model = re.sub(r'\blx l\b|\blxl\b', 'lx-l', model)
    model = re.sub(r'\btrd off road\b', 'trd off-road', model)
    model = re.sub(r'\bprius plug in\b|\bprius plug-in\b', 'prius prime', model)
    model = re.sub(r'\b2\.5s\b|\b2\.5 s\b', '2.5s', model)
    model = re.sub(r'\bwrangler unlimi.*\b|\bwrangler jk\b', 'wrangler unlimited', model)
    model = re.sub(r'\bf 150\b|\bf150\b', 'f-150', model)
    model = re.sub(r'\bf 250\b|\bf250\b', 'f-250', model)
    model = re.sub(r'\bf 350\b|\bf350\b', 'f-350', model)
    # --- 1. Collapse spaces in model numbers (put inside clean_model_name) ---
    model = re.sub(r'\b(rav|4runner|rx|es|is|gs|gx|lx|nx|e|f|f-)\s*(\d+)\b', r'\1\2', model)
    
    # --- 2. Update junk pattern to catch trailing body descriptions ---
    junk_pattern = r'\b(4d|4dr|2d|2dr|sedan|coupe|hatchback|wagon|suv|truck|pickup|clean|title|gas|diesel|hybrid|electric|sport utility|cargo van|cargo|passenger)\b'
    model = re.sub(junk_pattern, '', model)

    # --- Remove Junk Words (Keep valid trims like lx, gt, trd, rubicon) ---
    junk_pattern = r'\b(4d|4dr|2d|2dr|sedan|coupe|hatchback|wagon|suv|truck|pickup|clean|title|gas|diesel|hybrid|electric)\b'
    model = re.sub(junk_pattern, '', model)
    
    # Remove drivetrain tokens if you store AWD/4WD in a separate column
    model = re.sub(r'\b(4x4|awd|4wd|fwd|rwd|2wd)\b', '', model)
    
    # Clean up excess whitespace/hyphens
    model = re.sub(r'[\s\-]+', ' ', model).strip()
    
    return model if model else row['model']

# Run row-by-row normalization
df['model'] = df.apply(clean_model_name, axis=1)

# ============================================
# 4. FINAL POST-CLEANUP FILTERS (PUT NEW CODE HERE)
# ============================================

# Drop leftover cross-brand leaks that bypassed regex
LEAK_DROPS = {
    'chevrolet': ['e350', 'gladiator', 'savana 2500', 'grand prix', 'envoy', 'srx'],
    'gmc': ['e350', 'suburban', 'suburban 1500', 'suburban 2500', 'tahoe'],
    'dodge': ['sebring'],
    'kia': ['versa'],
    'lincoln': ['mountaineer'],
    'ram': ['titan'],
    'saturn': ['solstice'],
    'toyota': ['pilot ex l'],
    'mazda': ['m3']
}

for make, bad_models in LEAK_DROPS.items():
    df = df[~((df['manufacturer'] == make) & (df['model'].isin(bad_models)))]

# Drop orphan single digits and generic tokens missing real model names
GENERIC_TO_DROP = ['van', 'sport', '3', '5', '6', '1500', '2500', '3500', '5500', '2500hd']
df = df[~df['model'].isin(GENERIC_TO_DROP)]


# ============================================
# STEP 6 — De-duplicate & filter rare models
# ============================================
df = df.drop_duplicates()

# Keep models with sufficient sample counts for stable statistics
N = 30
model_counts = df['model'].value_counts()
valid_models = model_counts[model_counts >= N].index
df = df[df['model'].isin(valid_models)]

print(f"Unique canonical models remaining: {df['model'].nunique()}")

# Cap dataset size
df = df.sample(n=min(100000, len(df)), random_state=42).copy()
df.to_csv('../models/vehicles_clean.csv', index=False)


# ============================================
# STEP 7 — Feature Engineering (Ratios & Log Target)
# ============================================
from datetime import datetime

CURRENT_YEAR = datetime.now().year
df['age'] = (CURRENT_YEAR - df['year']).clip(lower=0)

# Engineered Ratio: Prevents low-mileage/high-age tree splitting anomalies
df['miles_per_year'] = df['odometer'] / (df['age'] + 1.0)

df = df.drop(columns=['year'])

# Log-transform target: forces XGBoost to optimize percentage error 
# instead of absolute dollar error (fixes economy car overvaluation)
df['log_price'] = np.log1p(df['price'])


# ============================================
# STEP 8 — Train/Test Split (STRICTLY BEFORE ENCODING)
# ============================================
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price', 'log_price'])
y_log = df['log_price']
y_raw = df['price']

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=42
)

print("Train split:", X_train.shape, "Test split:", X_test.shape)


# ============================================
# STEP 9 — Target Encoding Categorical Features
# ============================================
import category_encoders as ce

categorical_cols = ['manufacturer', 'model', 'fuel', 'transmission']

# TargetEncoder converts text models to continuous expected log-prices 
# without creating 1,200 sparse dummy columns
target_encoder = ce.TargetEncoder(cols=categorical_cols, smoothing=10)

# Fit ON TRAIN ONLY to completely prevent target leakage
X_train_encoded = target_encoder.fit_transform(X_train, y_train_log)
X_test_encoded = target_encoder.transform(X_test)


# ============================================
# STEP 10 — Train the XGBoost Model
# ============================================
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_encoded, y_train_log)


# ============================================
# STEP 11 — Evaluate Model (Inverting Log Scale)
# ============================================
from sklearn.metrics import mean_absolute_error, r2_score

y_pred_log = model.predict(X_test_encoded)
y_pred_raw = np.expm1(y_pred_log)  # Invert log1p back to actual USD

mae = mean_absolute_error(y_test_raw, y_pred_raw)
rmse = np.sqrt(np.mean((y_test_raw - y_pred_raw) ** 2))
r2 = r2_score(y_test_raw, y_pred_raw)

print(f"\n--- EVALUATION METRICS ---")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")


# ============================================
# STEP 12 — Extract Feature Importances
# ============================================
importances = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- FEATURE IMPORTANCES ---")
print(importances.to_string(index=False))


# ============================================
# STEP 13 — Save Model & Pipeline Artifacts
# ============================================
import joblib

joblib.dump(model, '../models/odomai_model.pkl')
joblib.dump(target_encoder, '../models/target_encoder.pkl')
joblib.dump(X_train_encoded.columns.tolist(), '../models/model_columns.pkl')

model.save_model('../models/odomai_model.json')

print("\nPipeline execution complete. All artifacts saved successfully.")

Artifacts saved above (`odomai_model.pkl`, `target_encoder.pkl`, `model_columns.pkl`) are exactly what `backend/app.py` loads at startup to serve `/predict`.

## 3. Sanity checks

Spot-check predictions against the cleaned dataset and inspect the manufacturer → model catalog used to populate the frontend's dropdowns (`/metadata` endpoint).

In [ ]:
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# Load saved pipeline artifacts
model = joblib.load('../models/odomai_model.pkl')
target_encoder = joblib.load('../models/target_encoder.pkl')
model_columns = joblib.load('../models/model_columns.pkl')

CURRENT_YEAR = datetime.now().year
df_clean = pd.read_csv('../models/vehicles_clean.csv')

test_rows = []
for model_name, group in df_clean.groupby('model'):
    age = max(0, CURRENT_YEAR - group['year'].median())
    odometer = group['odometer'].median()
    
    row = {
        'manufacturer': group['manufacturer'].mode()[0],
        'model': model_name,
        'fuel': group['fuel'].mode()[0],
        'transmission': group['transmission'].mode()[0],
        'age': age,
        'odometer': odometer,
        'miles_per_year': odometer / (age + 1.0)  # Calculate engineered ratio
    }
    test_rows.append(row)

test_df = pd.DataFrame(test_rows)

# Transform using the fitted TargetEncoder
test_encoded = target_encoder.transform(test_df)
test_encoded = test_encoded[model_columns]

# Predict log price and convert back to USD using expm1
log_preds = model.predict(test_encoded)
test_df['predicted_price'] = np.expm1(log_preds)

results = test_df[['manufacturer', 'model', 'age', 'odometer', 'predicted_price']] \
    .sort_values('predicted_price', ascending=False)

pd.set_option('display.max_rows', None)
print(results.to_string(index=False))

In [ ]:
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# Load saved pipeline artifacts
model = joblib.load('../models/odomai_model.pkl')
target_encoder = joblib.load('../models/target_encoder.pkl')
model_columns = joblib.load('../models/model_columns.pkl')

CURRENT_YEAR = datetime.now().year
df_clean = pd.read_csv('../models/vehicles_clean.csv')

age = 8 # Replace for testing
odometer = 63000 # Replace for testing
row = {
    'manufacturer': ['audi'], # Replace for testing
    'model': ['q5'], # Replace for testing
    'fuel': ['gas'], # Replace for testing
    'transmission': ['automatic'], # Replace for testing
    'age': age,
    'odometer': odometer,
    'miles_per_year': odometer / (age + 1.0)  # Calculate engineered ratio
}

test_df = pd.DataFrame(row)

# Transform using the fitted TargetEncoder
test_encoded = target_encoder.transform(test_df)
test_encoded = test_encoded[model_columns]

# Predict log price and convert back to USD using expm1
log_preds = model.predict(test_encoded)
test_df['predicted_price'] = np.expm1(log_preds)

results = test_df[['manufacturer', 'model', 'age', 'odometer', 'predicted_price']]
print(results.to_string(index=False))

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('../models/vehicles_clean.csv')

# 2. Group the data by manufacturer and get unique models
mfr_models = df.groupby('manufacturer')['model'].unique().apply(list).to_dict()

# 3. Print the total count and the breakdown for each manufacturer
print(f"Total manufacturers: {len(mfr_models)}\n")
for mfr, models in sorted(mfr_models.items()):
    print(f"{mfr.capitalize()} ({len(models)} models):")
    print(f"  {', '.join(models)}\n")